In [1]:
from datasets import Dataset

dataset_name = 'fineweb'
models = [
    ('clip', 'sentence-transformers/clip-ViT-B-32')
]

# Only load 1M samples for testing
dataset = Dataset.load_from_disk(f'embeddings/{dataset_name}/data')
sentences = dataset['text']

In [2]:
import torch
import numpy as np
from accelerate import Accelerator
from sentence_transformers import SentenceTransformer

acc = Accelerator()

for name, model_id in models:

    model = SentenceTransformer(model_id)
    model: SentenceTransformer = acc.prepare(model)

    pool = model.start_multi_process_pool(target_devices=['cuda:0', 'cuda:1', 'cuda:2', 'cuda:3'])
    embeddings = model.encode(sentences, pool=pool, batch_size=128, show_progress_bar=True, convert_to_numpy=True)
    model.stop_multi_process_pool(pool)

    np.save(f'embeddings/{dataset_name}/{name}.npy', embeddings, allow_pickle=True)

    # free up memory
    del model
    del embeddings
    torch.cuda.empty_cache()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Chunks:   0%|          | 0/200 [00:00<?, ?it/s]